# Feed-Forward Neural Language Model
## NLP Assignment 1 | BITS WILP — S2-25_AIMLCZG530

**Course:** S2-25_AIMLCZG530 — Natural Language Processing  
**Dataset:** AG News Classification Dataset (first 5,000 articles)  
**Objective:** Build and train a Feed-Forward Neural LM using PyTorch

| Component | Specification |
|---|---|
| Embedding Dim | 300 |
| Hidden Dim | 128 |
| Context Window | 3 words |
| Epochs | 10 |
| Min Vocab Freq | 3 |
| Optimizer | Adam (lr = 0.001) |
| Loss | Cross-Entropy |

---

## Cell 1: Configuration

> **Update `DATA_PATH`** to point to your downloaded AG News `train.csv` file.

In [1]:
# =============================================================================
# !! UPDATE DATA_PATH to point to your downloaded AG News CSV file !!
# =============================================================================
DATA_PATH: str = "train.csv"

## Cell 2: Imports & Logging Configuration

Import all required libraries and configure the `logging` module (replaces
bare `print` statements for pipeline flow messages).

In [2]:
import collections
import logging
import os
import random
import re
import sys
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ---------------------------------------------------------------------------
# Logging — replaces bare print() for all pipeline-flow messages
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

print(f"PyTorch version : {torch.__version__}")
print(f"Compute device  : {'CUDA — ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

PyTorch version : 2.10.0+cpu
Compute device  : CPU


---
## Section 1: Reproducibility

Fix random seeds for `random`, `numpy`, and `torch` (CPU & CUDA) to ensure
fully deterministic results across runs.

In [3]:
def set_seed(seed: int = 42) -> None:
    """Fix random seeds for full reproducibility across all libraries.

    Args:
        seed: Integer seed value applied globally to Python's ``random``
            module, NumPy, PyTorch (CPU and CUDA).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info("Random seed set to %d", seed)


set_seed(42)

2026-05-27 10:47:11 [INFO] __main__ — Random seed set to 42


---
## Section 2: Data Loading

Load the AG News CSV, resolve the text column case-insensitively
(`'Text'` or `'Description'`), and subset to the first **5,000** articles.

In [4]:
def load_ag_news(filepath: str, n_articles: int = 5000) -> pd.DataFrame:
    """Load the AG News CSV and return the first *n_articles* rows.

    Performs a case-insensitive column search for 'text' or 'description'
    to handle variations in the Kaggle AG News dataset format.

    Args:
        filepath: Path to the AG News .csv file.
        n_articles: Number of articles to subset (default: 5000).

    Returns:
        Single-column DataFrame with column name 'text'.

    Raises:
        FileNotFoundError: When the CSV does not exist at filepath.
        ValueError: When no usable text column is found.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Dataset not found at '{filepath}'. "
            "Download from https://www.kaggle.com/datasets/"
            "ilhamferdiona/ag-news-classification-dataset "
            "and update DATA_PATH at the top of this notebook."
        )

    logger.info("Loading dataset from '%s'", filepath)
    df = pd.read_csv(filepath, header=0)
    logger.info("Raw dataset shape: %s", df.shape)

    col_map: Dict[str, str] = {c.strip().lower(): c for c in df.columns}
    if "text" in col_map:
        text_col = col_map["text"]
    elif "description" in col_map:
        text_col = col_map["description"]
    else:
        raise ValueError(
            f"No suitable text column found. Available: {list(df.columns)}"
        )

    logger.info("Using column '%s' as text source", text_col)
    df = (
        df[[text_col]]
        .rename(columns={text_col: "text"})
        .head(n_articles)
        .copy()
    )
    df["text"] = df["text"].fillna("").astype(str)
    logger.info("Loaded %d articles", len(df))
    return df

---
## Section 3: Text Preprocessing

The `TextPreprocessor` class applies a sequential cleaning pipeline:

1. Lowercase conversion
2. Digit removal
3. Punctuation / special character removal
4. Whitespace normalisation
5. NLTK word tokenisation
6. English stopword removal
7. Single-character token removal

In [5]:
class TextPreprocessor:
    """Pipeline for cleaning and tokenising raw English text corpora.

    Attributes:
        _stopwords: Frozen set of English stopwords from NLTK.
        _num_pattern: Compiled regex matching digit sequences.
        _punct_pattern: Compiled regex matching non-word, non-space chars.
        _space_pattern: Compiled regex matching whitespace runs.
    """

    def __init__(self) -> None:
        """Download required NLTK corpora and compile regex patterns."""
        logger.info("Initialising TextPreprocessor — downloading NLTK resources")
        nltk.download("stopwords", quiet=True)
        nltk.download("punkt", quiet=True)
        nltk.download("punkt_tab", quiet=True)

        from nltk.corpus import stopwords as _sw
        self._stopwords: frozenset = frozenset(_sw.words("english"))
        self._num_pattern   = re.compile(r"\d+")
        self._punct_pattern = re.compile(r"[^\w\s]")
        self._space_pattern = re.compile(r"\s+")

    def clean(self, text: str) -> str:
        """Lowercase, remove digits/punctuation, normalise whitespace.

        Args:
            text: Raw input string.

        Returns:
            Cleaned lowercase string.
        """
        text = text.lower()
        text = self._num_pattern.sub(" ", text)
        text = self._punct_pattern.sub(" ", text)
        text = self._space_pattern.sub(" ", text).strip()
        return text

    def tokenize(self, text: str) -> List[str]:
        """Tokenise and filter stopwords from a cleaned string.

        Args:
            text: Pre-cleaned input string.

        Returns:
            List of filtered word tokens.
        """
        tokens = nltk.word_tokenize(text)
        return [t for t in tokens if t and len(t) > 1 and t not in self._stopwords]

    def process_corpus(self, texts: pd.Series) -> List[List[str]]:
        """Apply the full pipeline to a pandas Series of texts.

        Args:
            texts: Series of raw article strings.

        Returns:
            List of token lists, one per article.
        """
        logger.info("Preprocessing %d articles …", len(texts))
        tokenized = [self.tokenize(self.clean(doc)) for doc in texts]
        total = sum(len(t) for t in tokenized)
        logger.info("Preprocessing complete — total tokens: %d", total)
        return tokenized

---
## Section 4: Vocabulary

Build a vocabulary from the tokenised corpus enforcing `min_freq = 3`.

| Token | Index |
|---|---|
| `<PAD>` | 0 |
| `<UNK>` | 1 |
| most frequent word | 2 |
| second most frequent | 3 |
| … | … |

In [6]:
class Vocabulary:
    """Bidirectional word-index mapping with frequency filtering.

    Tokens below *min_freq* map to <UNK>.  Special tokens use fixed indices:
    <PAD> → 0, <UNK> → 1.  Regular tokens inserted in descending-frequency
    order for determinism.

    Attributes:
        PAD_TOKEN: '<PAD>' at index 0.
        UNK_TOKEN: '<UNK>' at index 1.
        min_freq: Minimum inclusion frequency.
        word2idx: Token → index mapping.
        idx2word: Index → token mapping.
        word_freq: Counter of raw token occurrences.
    """

    PAD_TOKEN: str = "<PAD>"
    UNK_TOKEN: str = "<UNK>"

    def __init__(self, min_freq: int = 3) -> None:
        """Initialise an empty vocabulary.

        Args:
            min_freq: Minimum corpus frequency for inclusion (default: 3).
        """
        self.min_freq  = min_freq
        self.word2idx: Dict[str, int]   = {}
        self.idx2word: Dict[int, str]   = {}
        self.word_freq: collections.Counter = collections.Counter()

    @property
    def vocab_size(self) -> int:
        """Total vocabulary size including special tokens."""
        return len(self.word2idx)

    def build(self, tokenized_corpus: List[List[str]]) -> None:
        """Construct vocabulary from tokenised corpus (descending-frequency order).

        Args:
            tokenized_corpus: List of token lists.
        """
        logger.info("Building vocabulary with min_freq=%d …", self.min_freq)
        for tokens in tokenized_corpus:
            self.word_freq.update(tokens)

        self.word2idx = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.idx2word = {0: self.PAD_TOKEN, 1: self.UNK_TOKEN}

        qualified = sorted(
            ((w, c) for w, c in self.word_freq.items() if c >= self.min_freq),
            key=lambda x: x[1],
            reverse=True,
        )
        for word, _ in qualified:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word

        logger.info(
            "Vocabulary built — %d qualified tokens (min_freq=%d); total size: %d",
            len(qualified), self.min_freq, self.vocab_size,
        )

    def encode(self, token: str) -> int:
        """Return index for token, falling back to <UNK>.

        Args:
            token: Input word.

        Returns:
            Integer index.
        """
        return self.word2idx.get(token, self.word2idx[self.UNK_TOKEN])

    def get_second_most_frequent_word(self) -> str:
        """Return the second most frequent word (excluding special tokens).

        Returns:
            Second most frequent word string.

        Raises:
            ValueError: If fewer than two qualified tokens exist.
        """
        qualified = [w for w in self.word2idx if w not in (self.PAD_TOKEN, self.UNK_TOKEN)]
        if len(qualified) < 2:
            raise ValueError("Vocabulary too small — cannot retrieve second most frequent word.")
        return qualified[1]  # index 0 = most frequent, index 1 = second most frequent

---
## Section 5: Dataset & DataLoader

Generate **(context, target)** pairs via a sliding window of size
`context_window + 1`:

$$\text{context} = [w_i,\; w_{i+1},\; w_{i+2}] \qquad \text{target} = w_{i+3}$$

Wrapped in a custom `torch.utils.data.Dataset` and fed through a
`DataLoader` with `batch_size = 512`.

In [7]:
class NGramDataset(Dataset):
    """PyTorch Dataset yielding n-gram context-target pairs.

    For context window k, each sample is:
        context : Tensor (k,)  — k consecutive token indices
        target  : scalar Tensor — index of the (k+1)-th token

    Attributes:
        context_window: Number of preceding tokens per context.
        data: List of (context_indices, target_index) tuples.
    """

    def __init__(
        self,
        tokenized_corpus: List[List[str]],
        vocab: Vocabulary,
        context_window: int = 3,
    ) -> None:
        """Generate all context-target pairs from the tokenised corpus.

        Args:
            tokenized_corpus: List of token lists, one per document.
            vocab: A fully built Vocabulary instance.
            context_window: Size of the context (default: 3).
        """
        self.context_window = context_window
        self.data: List[Tuple[List[int], int]] = []

        for tokens in tokenized_corpus:
            if len(tokens) <= context_window:
                continue
            encoded = [vocab.encode(t) for t in tokens]
            for i in range(len(encoded) - context_window):
                self.data.append((encoded[i: i + context_window], encoded[i + context_window]))

        logger.info(
            "NGramDataset — %d context-target pairs (context_window=%d)",
            len(self.data), context_window,
        )

    def __len__(self) -> int:
        """Return total number of (context, target) samples."""
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Retrieve one (context, target) pair.

        Args:
            idx: Sample index.

        Returns:
            Tuple of (LongTensor[context_window], scalar LongTensor).
        """
        context, target = self.data[idx]
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target,  dtype=torch.long),
        )


def build_dataloader(
    dataset: NGramDataset,
    batch_size: int = 512,
    shuffle: bool = True,
) -> DataLoader:
    """Wrap NGramDataset in a DataLoader.

    Args:
        dataset: Constructed NGramDataset.
        batch_size: Mini-batch size (default: 512).
        shuffle: Shuffle samples per epoch (default: True).

    Returns:
        Configured DataLoader.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)
    logger.info("DataLoader — %d batches of up to %d samples", len(loader), batch_size)
    return loader

---
## Section 6: Model Architecture

$$\text{Input}(B,\,C)
  \xrightarrow{\text{Embed}} (B,\,C,\,300)
  \xrightarrow{\text{flatten}} (B,\,900)
  \xrightarrow{\text{Linear}} (B,\,128)
  \xrightarrow{\text{ReLU}}
  \xrightarrow{\text{Linear}} (B,\,V)$$

| Layer | Input → Output |
|---|---|
| `nn.Embedding` | $(V, 300)$ table |
| Flatten | $(B, 3 \times 300 = 900)$ |
| `nn.Linear` (hidden) | $900 \rightarrow 128$ |
| `nn.ReLU` | — |
| `nn.Linear` (output) | $128 \rightarrow V$ |

In [8]:
class FeedForwardNLM(nn.Module):
    """Feed-Forward Neural Language Model.

    Architecture: Embedding → Flatten → Linear(900→128) → ReLU → Linear(128→V)

    Attributes:
        embedding: Learnable embedding table (V, embed_dim).
        fc1: Hidden fully-connected layer.
        relu: ReLU activation.
        fc2: Output projection layer mapping to vocabulary logits.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 300,
        hidden_dim: int = 128,
        context_window: int = 3,
        padding_idx: int = 0,
    ) -> None:
        """Initialise the Feed-Forward NLM.

        Args:
            vocab_size: Total vocabulary size.
            embed_dim: Word embedding dimensionality (default: 300).
            hidden_dim: Hidden layer neurons (default: 128).
            context_window: Context tokens per sample (default: 3).
            padding_idx: <PAD> index; gradient zeroed (default: 0).
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.fc1  = nn.Linear(context_window * embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(hidden_dim, vocab_size)

        logger.info(
            "FeedForwardNLM — vocab=%d | embed=%d | hidden=%d | ctx=%d | input_dim=%d",
            vocab_size, embed_dim, hidden_dim, context_window, context_window * embed_dim,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: LongTensor (B, context_window) — token indices.

        Returns:
            FloatTensor (B, vocab_size) — raw logits.
        """
        embedded  = self.embedding(x)                     # (B, C, E)
        flattened = embedded.view(embedded.size(0), -1)   # (B, C*E)
        hidden    = self.relu(self.fc1(flattened))        # (B, H)
        return self.fc2(hidden)                           # (B, V)

---
## Section 7: Trainer

The `Trainer` class encapsulates the full training loop:

- **Optimizer:** Adam (`lr = 0.001`)
- **Loss:** `nn.CrossEntropyLoss`
- **Epochs:** 10
- Per-epoch mean loss is logged via `logging.info` and stored for visualisation.

In [9]:
class Trainer:
    """Supervised training loop for FeedForwardNLM.

    Attributes:
        model: The neural language model.
        dataloader: Training DataLoader.
        device: Target compute device.
        optimizer: Adam optimizer.
        criterion: Cross-entropy loss function.
        epoch_losses: Mean loss per epoch (populated after train()).
    """

    def __init__(
        self,
        model: FeedForwardNLM,
        dataloader: DataLoader,
        device: torch.device,
        lr: float = 1e-3,
    ) -> None:
        """Initialise the Trainer.

        Args:
            model: FeedForwardNLM to optimise.
            dataloader: Training DataLoader.
            device: Target device (CPU or CUDA).
            lr: Adam learning rate (default: 0.001).
        """
        self.model       = model.to(device)
        self.dataloader  = dataloader
        self.device      = device
        self.optimizer   = torch.optim.Adam(model.parameters(), lr=lr)
        self.criterion   = nn.CrossEntropyLoss()
        self.epoch_losses: List[float] = []
        logger.info("Trainer — device=%s | lr=%g", device, lr)

    def train(self, n_epochs: int = 10) -> List[float]:
        """Run training for n_epochs passes over the data.

        Args:
            n_epochs: Number of epochs (default: 10).

        Returns:
            List of mean cross-entropy loss per epoch.
        """
        logger.info("Starting training — %d epochs", n_epochs)
        self.model.train()

        for epoch in range(1, n_epochs + 1):
            running_loss, n_batches = 0.0, 0

            for contexts, targets in self.dataloader:
                contexts = contexts.to(self.device)
                targets  = targets.to(self.device)

                self.optimizer.zero_grad()
                loss = self.criterion(self.model(contexts), targets)
                loss.backward()
                self.optimizer.step()

                running_loss += loss.item()
                n_batches    += 1

            mean_loss = running_loss / n_batches
            self.epoch_losses.append(mean_loss)
            logger.info("Epoch [%02d/%02d] — Mean Loss: %.6f", epoch, n_epochs, mean_loss)

        final = self.epoch_losses[-1]
        print(f"\n{'=' * 60}")
        print(f"  Training Complete")
        print(f"  Final Loss (Epoch {n_epochs}): {final:.6f}")
        print(f"{'=' * 60}\n")
        return self.epoch_losses

---
## Section 8: Visualisation — Training Loss Curve

Plots per-epoch mean cross-entropy loss using **Seaborn** and saves the
figure as `training_loss_curve.png`.

In [10]:
def plot_training_loss(
    losses: List[float],
    save_path: str = "training_loss_curve.png",
) -> None:
    """Plot per-epoch training loss and save to disk.

    Args:
        losses: List of mean loss values, one per epoch.
        save_path: Output PNG file path (default: 'training_loss_curve.png').
    """
    sns.set_theme(style="darkgrid", palette="muted")
    epochs = list(range(1, len(losses) + 1))

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.lineplot(
        x=epochs, y=losses, ax=ax,
        marker="o", linewidth=2.2, color="#2196F3", markersize=7,
        label="Training Loss",
    )
    ax.set_title(
        "Feed-Forward Neural Language Model — Training Loss Curve",
        fontsize=14, fontweight="bold", pad=14,
    )
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
    ax.set_xticks(epochs)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    logger.info("Training loss curve saved to '%s'", save_path)
    plt.show()

---
## Section 9: Theoretical Analysis

### Count-based (N-Gram) vs. Prediction-based (Neural LM) Embeddings

> Which technique better captures semantic relationships?
> Which performs better for rare words?

In [11]:
THEORETICAL_ANALYSIS: str = """
╬════════════════════════════════════════════════════════════════════════╣
║        THEORETICAL ANALYSIS — COUNT-BASED vs. NEURAL LM EMBEDDINGS     ║
╚════════════════════════════════════════════════════════════════════════╝

Count-based (N-Gram) methods construct word representations by tallying
co-occurrence statistics within a fixed context window, producing sparse,
high-dimensional vectors (e.g., PPMI matrices or TF-IDF variants).  These
vectors are directly interpretable and inexpensive to compute, but they
capture only shallow, surface-level co-occurrence patterns; two words that
rarely appear in identical contexts yield nearly orthogonal vectors even when
they are semantically synonymous (e.g., "automobile" vs. "car").  Because
every word's representation is derived solely from its own observed counts,
rare words accumulate insufficient statistical evidence to form reliable
embeddings — their vectors are noisy, underspecified, and fail to encode
meaningful semantic neighbourhoods.

Prediction-based (Neural LM) methods, exemplified by the Feed-Forward model
implemented here, learn dense, low-dimensional embeddings by optimising a
predictive objective: the model must correctly forecast a target word from its
surrounding context.  During back-propagation the gradient signal updates each
word's embedding by propagating information from semantically related contexts,
enabling generalisation across words that share distributional properties.
Crucially, this architecture handles rare words more gracefully than
count-based methods: through shared hidden-layer parameters, a rare word whose
few occurrences co-occur with common, well-trained words can still acquire a
meaningful dense vector via indirect gradient flow.  Furthermore, the
continuous embedding space encodes graded analogical and synonym relationships
that co-occurrence matrices cannot represent (e.g., king − man + woman ≈ queen).

Verdict: Prediction-based Neural LM embeddings substantially outperform
count-based N-Gram embeddings for rare words, because the neural objective
function allows gradient information to flow from high-frequency neighbours
into rare-word representations, yielding denser, semantically richer vectors
even from limited training signal.
"""

---
## Section 10: Execution Pipeline

Run each step cell below **in order** to execute the full end-to-end pipeline.

### Step 1 — Seed & Device Detection

In [12]:
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Compute device: %s", device)

2026-05-27 10:47:12 [INFO] __main__ — Random seed set to 42
2026-05-27 10:47:12 [INFO] __main__ — Compute device: cpu


### Step 2 — Load AG News Data

In [13]:
df = load_ag_news(DATA_PATH, n_articles=5000)
print(f"Shape : {df.shape}")
df.head()

2026-05-27 10:47:12 [INFO] __main__ — Loading dataset from 'train.csv'
2026-05-27 10:47:13 [INFO] __main__ — Raw dataset shape: (120000, 3)
2026-05-27 10:47:13 [INFO] __main__ — Using column 'Description' as text source
2026-05-27 10:47:13 [INFO] __main__ — Loaded 5000 articles
Shape : (5000, 1)


,text
0,"Reuters - Short-sellers, Wall Street's dwindli..."
1,Reuters - Private investment firm Carlyle Grou...
2,Reuters - Soaring crude prices plus worries\ab...
3,Reuters - Authorities have halted oil export\f...
4,"AFP - Tearaway world oil prices, toppling reco..."


### Step 3 — Text Preprocessing

In [14]:
preprocessor     = TextPreprocessor()
tokenized_corpus = preprocessor.process_corpus(df["text"])

print(f"Total documents   : {len(tokenized_corpus)}")
print(f"Sample tokens (0) : {tokenized_corpus[0][:15]}")

2026-05-27 10:47:14 [INFO] __main__ — Initialising TextPreprocessor — downloading NLTK resources
2026-05-27 10:47:17 [INFO] __main__ — Preprocessing 5000 articles …
2026-05-27 10:47:18 [INFO] __main__ — Preprocessing complete — total tokens: 105665
Total documents   : 5000
Sample tokens (0) : ['reuters', 'short', 'sellers', 'wall', 'street', 'dwindling', 'band', 'ultra', 'cynics', 'seeing', 'green']


### Step 4 — Build Vocabulary (min_freq = 3)

In [15]:
vocab = Vocabulary(min_freq=3)
vocab.build(tokenized_corpus)

print(f"Vocabulary size       : {vocab.vocab_size}")
print(f"Most frequent word    : '{vocab.idx2word[2]}'  (freq={vocab.word_freq[vocab.idx2word[2]]})")
print(f"2nd most frequent     : '{vocab.idx2word[3]}'  (freq={vocab.word_freq[vocab.idx2word[3]]})")

2026-05-27 10:47:18 [INFO] __main__ — Building vocabulary with min_freq=3 …
2026-05-27 10:47:18 [INFO] __main__ — Vocabulary built — 5938 qualified tokens (min_freq=3); total size: 5940
Vocabulary size       : 5940
Most frequent word    : 'reuters'  (freq=1119)
2nd most frequent     : 'said'  (freq=835)


### Step 5 — Dataset & DataLoader

In [16]:
dataset = NGramDataset(tokenized_corpus, vocab, context_window=3)
if len(dataset) == 0:
    raise RuntimeError(
        "Dataset is empty — check that DATA_PATH points to a valid AG News CSV."
    )

dataloader = build_dataloader(dataset, batch_size=512, shuffle=True)

print(f"Total samples  : {len(dataset):,}")
print(f"Total batches  : {len(dataloader)}")

sample_ctx, sample_tgt = dataset[0]
print(f"\nSample context indices : {sample_ctx.tolist()}")
print(f"Sample context words   : {[vocab.idx2word[i] for i in sample_ctx.tolist()]}")
print(f"Sample target word     : '{vocab.idx2word[sample_tgt.item()]}'")

2026-05-27 10:47:19 [INFO] __main__ — NGramDataset — 90665 context-target pairs (context_window=3)
2026-05-27 10:47:19 [INFO] __main__ — DataLoader — 178 batches of up to 512 samples
Total samples  : 90,665
Total batches  : 178

Sample context indices : [2, 641, 2949]
Sample context words   : ['reuters', 'short', 'sellers']
Sample target word     : 'wall'


### Step 6 — Instantiate Model

In [17]:
model = FeedForwardNLM(
    vocab_size     = vocab.vocab_size,
    embed_dim      = 300,
    hidden_dim     = 128,
    context_window = 3,
    padding_idx    = vocab.word2idx[Vocabulary.PAD_TOKEN],
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
print(model)

2026-05-27 10:47:19 [INFO] __main__ — FeedForwardNLM — vocab=5940 | embed=300 | hidden=128 | ctx=3 | input_dim=900
Total trainable parameters: 2,663,588
FeedForwardNLM(
  (embedding): Embedding(5940, 300, padding_idx=0)
  (fc1): Linear(in_features=900, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=5940, bias=True)
)


### Step 7 — Train for 10 Epochs

In [ ]:
trainer      = Trainer(model, dataloader, device, lr=1e-3)
epoch_losses = trainer.train(n_epochs=25)

2026-05-27 10:47:23 [INFO] __main__ — Trainer — device=cpu | lr=0.001
2026-05-27 10:47:23 [INFO] __main__ — Starting training — 25 epochs
2026-05-27 10:48:00 [INFO] __main__ — Epoch [01/25] — Mean Loss: 7.400649
2026-05-27 10:48:44 [INFO] __main__ — Epoch [02/25] — Mean Loss: 6.275258
2026-05-27 10:49:36 [INFO] __main__ — Epoch [03/25] — Mean Loss: 5.411457
2026-05-27 10:50:11 [INFO] __main__ — Epoch [04/25] — Mean Loss: 4.501267
2026-05-27 10:50:59 [INFO] __main__ — Epoch [05/25] — Mean Loss: 3.745390
2026-05-27 10:52:07 [INFO] __main__ — Epoch [06/25] — Mean Loss: 3.182931
2026-05-27 10:52:56 [INFO] __main__ — Epoch [07/25] — Mean Loss: 2.749612
2026-05-27 10:53:52 [INFO] __main__ — Epoch [08/25] — Mean Loss: 2.422901
2026-05-27 10:54:28 [INFO] __main__ — Epoch [09/25] — Mean Loss: 2.156536
2026-05-27 10:55:02 [INFO] __main__ — Epoch [10/25] — Mean Loss: 1.939988
2026-05-27 10:55:34 [INFO] __main__ — Epoch [11/25] — Mean Loss: 1.750078
2026-05-27 10:56:04 [INFO] __main__ — Epoch [12/

### Step 8 — Learned Embedding for Second Most Frequent Word

In [ ]:
second_word      : str       = vocab.get_second_most_frequent_word()
second_word_idx  : int       = vocab.encode(second_word)
embedding_vector : np.ndarray = (
    model.embedding.weight[second_word_idx].detach().cpu().numpy()
)

print(f"{'─' * 60}")
print(f"  Second Most Frequent Word  : '{second_word}'")
print(f"  Corpus Frequency           : {vocab.word_freq[second_word]}")
print(f"  Vocabulary Index           : {second_word_idx}")
print(f"  Embedding Vector Shape     : {embedding_vector.shape}")
print(f"  First 10 Dimensions        : {np.round(embedding_vector[:10], 6)}")
print(f"\n  Full Embedding Vector ({embedding_vector.shape[0]}D):")
print(f"  {embedding_vector}")
print(f"{'─' * 60}")

### Step 9 — Training Loss Curve

In [ ]:
plot_training_loss(epoch_losses, save_path="training_loss_curve.png")

### Step 10 — Theoretical Analysis

In [ ]:
print(THEORETICAL_ANALYSIS)